# Intelligent Image Recognition System

Beginner notebook using TensorFlow/Keras and MobileNetV2. Run cells from top to bottom. Upload your dataset ZIP when prompted. Expected ZIP structure: `dataset/train/class_name`, `dataset/validation/class_name`, and `dataset/test/class_name`.

## 1. Install/import libraries

In [ ]:
%pip -q install tensorflow scikit-learn matplotlib pillow

In [ ]:
import os, zipfile, json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
print('TensorFlow version:', tf.__version__)

## 2. Upload and extract your dataset ZIP

Prepare a ZIP containing `dataset/train`, `dataset/validation`, and `dataset/test`. Each split must contain one folder per class. If your ZIP has a different folder layout, adjust the paths below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select your dataset ZIP
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content')
DATA_DIR = '/content/dataset'
print('Dataset folders:', os.listdir(DATA_DIR))

## 3. Load the image data

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

def load_split(path, training=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        path, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        label_mode='int', shuffle=training, seed=42)
    names = ds.class_names
    if training:
        aug = tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.05), layers.RandomZoom(0.10)])
        ds = ds.map(lambda x, y: (preprocess_input(aug(tf.cast(x, tf.float32), training=True)), y), num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE), names

train_ds, class_names = load_split(f'{DATA_DIR}/train', training=True)
val_ds, val_names = load_split(f'{DATA_DIR}/validation')
test_ds, test_names = load_split(f'{DATA_DIR}/test')
assert class_names == val_names == test_names, 'Class folders must match across splits.'
print('Classes:', class_names)

## 4. Build the MobileNetV2 model

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)
model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Train the model

In [ ]:
EPOCHS = 5  # You can increase this after the first successful run
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

## 6. Plot accuracy and loss

In [ ]:
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

## 7. Evaluate using unseen test images

Use the real printed values in your internship report. Do not invent metrics.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
all_y = np.concatenate([y.numpy() for _, y in test_ds])
probs = model.predict(test_ds)
pred_y = np.argmax(probs, axis=1)
print('Test accuracy:', round(float(test_accuracy), 4))
print(classification_report(all_y, pred_y, labels=range(len(class_names)), target_names=class_names, zero_division=0))
cm = confusion_matrix(all_y, pred_y, labels=range(len(class_names)))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(xticks_rotation=45)
plt.tight_layout(); plt.show()

## 8. Save the trained model

In [ ]:
model.save('/content/image_recognition_model.keras')
with open('/content/class_names.json', 'w') as f:
    json.dump(class_names, f)
print('Model and class names saved.')
files.download('/content/image_recognition_model.keras')
files.download('/content/class_names.json')

## 9. Predict a new image

In [ ]:
new_upload = files.upload()  # Select one image not used for training/testing
from tensorflow.keras.utils import load_img, img_to_array
img_path = next(iter(new_upload))
img = load_img(img_path, target_size=IMG_SIZE)
arr = preprocess_input(np.expand_dims(img_to_array(img), axis=0))
probabilities = model.predict(arr, verbose=0)[0]
idx = int(np.argmax(probabilities))
print('Predicted class:', class_names[idx])
print(f'Confidence: {probabilities[idx]*100:.2f}%')